In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

In [2]:
from agent.config_models import AgentSettings, ReviewMode, ReviewRequest
from agent.runtime import run_agent, AgentResult
from prompts.orchestrator import ORCHESTRATOR_PROMPT
from utils import load_artifact


def notebook_ask_user(input_data: dict) -> dict[str, str]:
    answers = {}
    for q in input_data.get("questions", []):
        print(f"\n{q['question']}")
        options = q.get("options", [])
        for i, opt in enumerate(options):
            print(f"  {i+1}. {opt['label']} — {opt['description']}")
        response = input("Your answer: ").strip()
        parsed_labels = []
        if options:
            indices = [part.strip() for part in response.split(",") if part.strip()]
            if indices and all(part.isdigit() for part in indices):
                for part in indices:
                    index = int(part) - 1
                    if 0 <= index < len(options):
                        parsed_labels.append(options[index]["label"])
        answers[q["question"]] = ", ".join(parsed_labels) if parsed_labels else response
    return answers

In [ ]:
request = ReviewRequest(mode=ReviewMode.STANDARD)
settings = AgentSettings.from_request(request)

print(f"Mode: {settings.mode}")
print(f"Model: {settings.model}")
print(f"Workspace: {settings.workspace}")
print(f"Subagents: {settings.subagent_count}")
print(f"Self-play rounds: {settings.self_play_rounds}")

In [ ]:
artifact = load_artifact("../paper_clean_final.pdf")
print(f"Title: {artifact.title}")
print(f"Length: {len(artifact.text)} chars")

In [ ]:
prompt = ORCHESTRATOR_PROMPT.format(self_play_rounds=settings.self_play_rounds)

result = await run_agent(
    system_prompt=prompt,
    user_prompt=f"Review this paper:\n\n{artifact.text}",
    model=settings.model,
    cwd=settings.workspace,
    ask_user=notebook_ask_user,
)
print(f"Session ID: {result.session_id}")

In [ ]:
from IPython.display import Markdown
Markdown(result.text)